In [0]:
# ============================================================
# Notebook  : ingest_flight_prices
# Purpose   : Fetch flight prices from AeroDataBox API
#             for all active routes and append to Bronze table
# Author    : FlightPriceTracker
# Version   : v1.0
# ============================================================

import uuid
import hashlib
import json
import time
import random
import requests
import traceback
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, TimestampType
)
from delta.tables import DeltaTable

# ============================================================
# SECTION 1 — INIT
# ============================================================

spark = SparkSession.builder \
    .appName("FlightPriceTracker_Ingest") \
    .getOrCreate()

# ============================================================
# SECTION 2 — CONSTANTS
# ============================================================

RUN_ID             = str(uuid.uuid4())
JOB_NAME           = "FlightPriceTracker"
NOTEBOOK_NAME      = "ingest_flight_prices"
TASK_SEQUENCE      = 1
TRIGGERED_BY       = "SCHEDULER"
RAPIDAPI_KEY       = "0c38577044msh5d0fec55efb6c5dp179d02jsn1785120c0a3a"
RAPIDAPI_HOST      = "aerodatabox.p.rapidapi.com"
API_BASE_URL       = "https://aerodatabox.p.rapidapi.com/flights/airports/iata/{origin}"
API_VERSION        = "v1"
SOURCE_API         = "aerodatabox"
MAX_RETRIES        = 3
RETRY_WAIT_SECONDS = 60
ROUTE_WAIT_SECONDS = 5
DATABASE           = "workspace.flight_tracker"
RAW_TABLE          = f"{DATABASE}.flight_prices_raw"
AUDIT_TABLE        = f"{DATABASE}.pipeline_audit_log"
ERROR_TABLE        = f"{DATABASE}.api_error_log"
ROUTES_TABLE       = f"{DATABASE}.routes_config"

# Base prices per route — realistic INR converted to USD
ROUTE_BASE_PRICES = {
    "DEL-BOM" : 120.0,
    "DEL-BLR" : 110.0,
    "BOM-MAA" : 95.0,
    "CCU-DEL" : 100.0,
    "DEL-DXB" : 280.0,
    "BOM-LHR" : 650.0,
    "DEL-SIN" : 380.0,
    "BOM-JFK" : 820.0,
}

def mock_price(route_id):
    base       = ROUTE_BASE_PRICES.get(route_id, 200.0)
    variation  = random.uniform(-0.30, 0.10)
    price      = round(base * (1 + variation), 2)
    return max(price, 50.0)

def now():
    return datetime.now(timezone.utc)

print(f"RUN_ID     : {RUN_ID}")
print(f"NOTEBOOK   : {NOTEBOOK_NAME}")
print(f"START TIME : {now()}")

# ============================================================
# SECTION 3 — SCHEMAS
# ============================================================

audit_schema = StructType([
    StructField("run_id",               StringType(),    True),
    StructField("job_name",             StringType(),    True),
    StructField("notebook_name",        StringType(),    True),
    StructField("task_sequence",        IntegerType(),   True),
    StructField("cluster_id",           StringType(),    True),
    StructField("start_time",           TimestampType(), True),
    StructField("end_time",             TimestampType(), True),
    StructField("duration_seconds",     IntegerType(),   True),
    StructField("status",               StringType(),    True),
    StructField("records_read",         IntegerType(),   True),
    StructField("records_written",      IntegerType(),   True),
    StructField("records_skipped",      IntegerType(),   True),
    StructField("records_failed",       IntegerType(),   True),
    StructField("error_message",        StringType(),    True),
    StructField("error_stacktrace",     StringType(),    True),
    StructField("retry_attempt",        IntegerType(),   True),
    StructField("triggered_by",         StringType(),    True),
    StructField("databricks_job_run_id",StringType(),    True),
    StructField("created_at",           TimestampType(), True),
])

error_schema = StructType([
    StructField("error_id",           StringType(),    True),
    StructField("run_id",             StringType(),    True),
    StructField("route_id",           StringType(),    True),
    StructField("api_endpoint",       StringType(),    True),
    StructField("http_status_code",   IntegerType(),   True),
    StructField("error_code",         StringType(),    True),
    StructField("error_message",      StringType(),    True),
    StructField("request_payload",    StringType(),    True),
    StructField("response_payload",   StringType(),    True),
    StructField("retry_attempt",      IntegerType(),   True),
    StructField("max_retries",        IntegerType(),   True),
    StructField("resolved",           BooleanType(),   True),
    StructField("resolved_at",        TimestampType(), True),
    StructField("failed_at",          TimestampType(), True),
])

raw_schema = StructType([
    StructField("run_id",               StringType(),    True),
    StructField("route_id",             StringType(),    True),
    StructField("origin",               StringType(),    True),
    StructField("destination",          StringType(),    True),
    StructField("airline",              StringType(),    True),
    StructField("airline_code",         StringType(),    True),
    StructField("aircraft_type",        StringType(),    True),
    StructField("flight_number",        StringType(),    True),
    StructField("num_stops",            IntegerType(),   True),
    StructField("stop_locations",       StringType(),    True),
    StructField("departure_datetime",   TimestampType(), True),
    StructField("arrival_datetime",     TimestampType(), True),
    StructField("duration_minutes",     IntegerType(),   True),
    StructField("price_usd",            DoubleType(),    True),
    StructField("cabin_class",          StringType(),    True),
    StructField("seats_available",      IntegerType(),   True),
    StructField("baggage_allowance_kg", IntegerType(),   True),
    StructField("is_refundable",        BooleanType(),   True),
    StructField("source_api",           StringType(),    True),
    StructField("api_response_code",    IntegerType(),   True),
    StructField("api_version",          StringType(),    True),
    StructField("raw_payload",          StringType(),    True),
    StructField("ingested_at",          TimestampType(), True),
    StructField("record_checksum",      StringType(),    True),
])

# ============================================================
# SECTION 4 — AUDIT HELPER
# ============================================================

def log_audit(
    run_id, job_name, notebook_name, task_sequence,
    start_time, end_time, status,
    records_read=0, records_written=0,
    records_skipped=0, records_failed=0,
    error_message=None, error_stacktrace=None,
    retry_attempt=0, triggered_by="SCHEDULER"
):
    duration = int((end_time - start_time).total_seconds())
    audit_data = [{
        "run_id"               : run_id,
        "job_name"             : job_name,
        "notebook_name"        : notebook_name,
        "task_sequence"        : task_sequence,
        "cluster_id"           : "serverless",
        "start_time"           : start_time,
        "end_time"             : end_time,
        "duration_seconds"     : duration,
        "status"               : status,
        "records_read"         : records_read,
        "records_written"      : records_written,
        "records_skipped"      : records_skipped,
        "records_failed"       : records_failed,
        "error_message"        : error_message if error_message else "",
        "error_stacktrace"     : error_stacktrace if error_stacktrace else "",
        "retry_attempt"        : retry_attempt,
        "triggered_by"         : triggered_by,
        "databricks_job_run_id": "",
        "created_at"           : now()
    }]
    audit_df = spark.createDataFrame(audit_data, schema=audit_schema)
    audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)
    print(f"[AUDIT] {status} logged for {notebook_name}")

# ============================================================
# SECTION 5 — API ERROR HELPER
# ============================================================

def log_api_error(
    run_id, route_id, api_endpoint,
    http_status_code, error_code, error_message,
    request_payload, response_payload,
    retry_attempt, resolved=False
):
    error_data = [{
        "error_id"          : str(uuid.uuid4()),
        "run_id"            : run_id,
        "route_id"          : route_id,
        "api_endpoint"      : api_endpoint,
        "http_status_code"  : http_status_code,
        "error_code"        : error_code,
        "error_message"     : error_message,
        "request_payload"   : json.dumps(request_payload),
        "response_payload"  : response_payload if response_payload else "",
        "retry_attempt"     : retry_attempt,
        "max_retries"       : MAX_RETRIES,
        "resolved"          : resolved,
        "resolved_at"       : None,
        "failed_at"         : now()
    }]
    error_df = spark.createDataFrame(error_data, schema=error_schema)
    error_df.write.format("delta").mode("append").saveAsTable(ERROR_TABLE)
    print(f"[ERROR LOG] Logged for route {route_id} — attempt {retry_attempt}")

# ============================================================
# SECTION 6 — CHECKSUM HELPER
# ============================================================

def compute_checksum(route_id, airline, flight_number, departure_datetime, price_usd):
    raw = f"{route_id}|{airline}|{flight_number}|{departure_datetime}|{price_usd}"
    return hashlib.md5(raw.encode()).hexdigest()

# ============================================================
# SECTION 7 — API FETCH WITH RETRY
# ============================================================

def fetch_flights(route_id, origin, destination):
    url     = API_BASE_URL.format(origin=origin)
    headers = {
        "X-RapidAPI-Key"  : RAPIDAPI_KEY,
        "X-RapidAPI-Host" : RAPIDAPI_HOST
    }
    params  = {
        "offsetMinutes"   : 0,
        "durationMinutes" : 720,
        "direction"       : "Departure",
        "withLeg"         : True,
        "withCancelled"   : False,
        "withCodeshared"  : False,
        "withLocation"    : False
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"[FETCH] {route_id} — Attempt {attempt}")
            response = requests.get(url, headers=headers, params=params, timeout=30)

            if response.status_code == 200:
                data       = response.json()
                departures = data.get("departures", [])
                if departures:
                    print(f"[FETCH] {route_id} — Success. Records: {len(departures)}")
                    return response.status_code, departures, None
                print(f"[FETCH] {route_id} — Empty response from API")
                return response.status_code, None, "Empty departures in response"

            error_msg = f"HTTP {response.status_code} — {response.text}"
            print(f"[FETCH] {route_id} — Failed: {error_msg}")
            log_api_error(
                run_id           = RUN_ID,
                route_id         = route_id,
                api_endpoint     = url,
                http_status_code = response.status_code,
                error_code       = str(response.status_code),
                error_message    = error_msg,
                request_payload  = params,
                response_payload = response.text,
                retry_attempt    = attempt
            )

        except requests.exceptions.Timeout:
            print(f"[FETCH] {route_id} — Timeout on attempt {attempt}")
            log_api_error(
                run_id           = RUN_ID,
                route_id         = route_id,
                api_endpoint     = url,
                http_status_code = 408,
                error_code       = "TIMEOUT",
                error_message    = "Request timed out",
                request_payload  = params,
                response_payload = "",
                retry_attempt    = attempt
            )

        except Exception as e:
            print(f"[FETCH] {route_id} — Exception: {str(e)}")
            log_api_error(
                run_id           = RUN_ID,
                route_id         = route_id,
                api_endpoint     = url,
                http_status_code = 500,
                error_code       = "EXCEPTION",
                error_message    = str(e),
                request_payload  = params,
                response_payload = "",
                retry_attempt    = attempt
            )

        if attempt < MAX_RETRIES:
            print(f"[RETRY] Waiting {RETRY_WAIT_SECONDS}s...")
            time.sleep(RETRY_WAIT_SECONDS)

    return None, None, f"All {MAX_RETRIES} retries exhausted for {route_id}"

# ============================================================
# SECTION 8 — PARSE FLIGHT RECORD
# ============================================================

def parse_flight(route_id, origin, destination, flight, status_code):
    try:
        dep          = flight.get("departure", {})
        arr          = flight.get("arrival", {})
        airline      = flight.get("airline", {})
        aircraft     = flight.get("aircraft", {})
        fn           = flight.get("number", "UNKNOWN")
        dep_dt       = dep.get("scheduledTime", {}).get("utc")
        arr_dt       = arr.get("scheduledTime", {}).get("utc")
        airline_name = airline.get("name", "UNKNOWN")
        airline_code = airline.get("iata", "UNKNOWN")
        price        = mock_price(route_id)

        dep_dt_parsed = datetime.strptime(dep_dt, "%Y-%m-%d %H:%M%z") if dep_dt else None
        arr_dt_parsed = datetime.strptime(arr_dt, "%Y-%m-%d %H:%M%z") if arr_dt else None
        duration      = int((arr_dt_parsed - dep_dt_parsed).total_seconds() / 60) \
                        if dep_dt_parsed and arr_dt_parsed else None

        return {
            "run_id"               : RUN_ID,
            "route_id"             : route_id,
            "origin"               : origin,
            "destination"          : destination,
            "airline"              : airline_name,
            "airline_code"         : airline_code,
            "aircraft_type"        : aircraft.get("model", ""),
            "flight_number"        : fn,
            "num_stops"            : 0,
            "stop_locations"       : "",
            "departure_datetime"   : dep_dt_parsed,
            "arrival_datetime"     : arr_dt_parsed,
            "duration_minutes"     : duration,
            "price_usd"            : float(price),
            "cabin_class"          : "ECONOMY",
            "seats_available"      : random.randint(1, 180),
            "baggage_allowance_kg" : 15,
            "is_refundable"        : False,
            "source_api"           : SOURCE_API,
            "api_response_code"    : status_code,
            "api_version"          : API_VERSION,
            "raw_payload"          : json.dumps(flight),
            "ingested_at"          : now(),
            "record_checksum"      : compute_checksum(route_id, airline_name, fn, str(dep_dt_parsed), price)
        }

    except Exception as e:
        print(f"[PARSE] Failed for route {route_id}: {str(e)}")
        return None

# ============================================================
# SECTION 9 — MAIN EXECUTION
# ============================================================

start_time      = now()
all_records     = []
records_read    = 0
records_written = 0
records_skipped = 0
records_failed  = 0
pipeline_status = "SUCCESS"
pipeline_error  = None
pipeline_trace  = None

try:
    print(f"\n[ROUTES] Reading active routes from {ROUTES_TABLE}...")
    routes_df = spark.sql(f"SELECT route_id, origin, destination FROM {ROUTES_TABLE} WHERE is_active = true")
    routes    = routes_df.collect()
    print(f"[ROUTES] Active routes found: {len(routes)}")

    for row in routes:
        route_id    = row["route_id"]
        origin      = row["origin"]
        destination = row["destination"]

        print(f"\n[PROCESS] Route: {route_id}")
        status_code, departures, error = fetch_flights(route_id, origin, destination)

        if error or departures is None:
            print(f"[SKIP] {route_id} — {error}")
            records_skipped += 1
            time.sleep(ROUTE_WAIT_SECONDS)
            continue

        records_read += len(departures)

        for flight in departures:
            arr_iata = flight.get("arrival", {}).get("airport", {}).get("iata", "")
            if arr_iata != destination:
                continue
            parsed = parse_flight(route_id, origin, destination, flight, status_code)
            if parsed:
                all_records.append(parsed)
            else:
                records_failed += 1

        time.sleep(ROUTE_WAIT_SECONDS)

    if all_records:
        print(f"\n[WRITE] Writing {len(all_records)} records to {RAW_TABLE}...")
        raw_df = spark.createDataFrame(all_records, schema=raw_schema)
        raw_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(RAW_TABLE)
        records_written = len(all_records)
        print(f"[WRITE] Done. Records written: {records_written}")
    else:
        print("[WRITE] No records to write.")
        pipeline_status = "PARTIAL"

except Exception as e:
    pipeline_status = "FAILED"
    pipeline_error  = str(e)
    pipeline_trace  = traceback.format_exc()
    print(f"[ERROR] {pipeline_error}")

finally:
    end_time = now()
    log_audit(
        run_id           = RUN_ID,
        job_name         = JOB_NAME,
        notebook_name    = NOTEBOOK_NAME,
        task_sequence    = TASK_SEQUENCE,
        start_time       = start_time,
        end_time         = end_time,
        status           = pipeline_status,
        records_read     = records_read,
        records_written  = records_written,
        records_skipped  = records_skipped,
        records_failed   = records_failed,
        error_message    = pipeline_error,
        error_stacktrace = pipeline_trace,
        triggered_by     = TRIGGERED_BY
    )
    print(f"\n[DONE] {NOTEBOOK_NAME} — Status: {pipeline_status}")
    print(f"[DONE] Read: {records_read} | Written: {records_written} | Skipped: {records_skipped} | Failed: {records_failed}")
    dbutils.notebook.exit(RUN_ID)

In [0]:
%sql
SELECT * FROM workspace.flight_tracker.flight_prices_raw;

run_id,route_id,origin,destination,airline,airline_code,aircraft_type,flight_number,num_stops,stop_locations,departure_datetime,arrival_datetime,duration_minutes,price_usd,cabin_class,seats_available,baggage_allowance_kg,is_refundable,source_api,api_response_code,api_version,raw_payload,ingested_at,record_checksum
bd8bf4e5-a891-4e3e-aea4-adf8386b3bd4,DEL-BLR,DEL,BLR,IndiGo,6E,Airbus A320,6E 811,0,,2026-06-29T12:50:00.000Z,2026-06-29T15:40:00.000Z,170,83.48,ECONOMY,162,15,false,aerodatabox,200,v1,"{""departure"": {""scheduledTime"": {""utc"": ""2026-06-29 12:50Z"", ""local"": ""2026-06-29 18:20+05:30""}, ""revisedTime"": {""utc"": ""2026-06-29 12:50Z"", ""local"": ""2026-06-29 18:20+05:30""}, ""terminal"": ""1"", ""gate"": ""D20"", ""quality"": [""Basic"", ""Live""]}, ""arrival"": {""airport"": {""icao"": ""VOBL"", ""iata"": ""BLR"", ""name"": ""Bangalore"", ""countryCode"": ""in"", ""timeZone"": ""Asia/Kolkata""}, ""scheduledTime"": {""utc"": ""2026-06-29 15:40Z"", ""local"": ""2026-06-29 21:10+05:30""}, ""revisedTime"": {""utc"": ""2026-06-29 15:40Z"", ""local"": ""2026-06-29 21:10+05:30""}, ""terminal"": ""1"", ""gate"": ""A1"", ""baggageBelt"": ""1"", ""quality"": [""Basic"", ""Live""]}, ""number"": ""6E 811"", ""status"": ""CheckIn"", ""codeshareStatus"": ""IsOperator"", ""isCargo"": false, ""aircraft"": {""reg"": ""VT-ICM"", ""modeS"": ""80173E"", ""model"": ""Airbus A320""}, ""airline"": {""name"": ""IndiGo"", ""iata"": ""6E"", ""icao"": ""IGO""}}",2026-06-29T12:05:13.248Z,7f57881df6828d302ab191721d9138b0
bd8bf4e5-a891-4e3e-aea4-adf8386b3bd4,DEL-BLR,DEL,BLR,Air India,AI,Airbus A320 NEO,AI 2487,0,,2026-06-29T13:00:00.000Z,2026-06-29T15:55:00.000Z,175,80.45,ECONOMY,47,15,false,aerodatabox,200,v1,"{""departure"": {""scheduledTime"": {""utc"": ""2026-06-29 13:00Z"", ""local"": ""2026-06-29 18:30+05:30""}, ""revisedTime"": {""utc"": ""2026-06-29 13:00Z"", ""local"": ""2026-06-29 18:30+05:30""}, ""terminal"": ""3"", ""gate"": ""51"", ""quality"": [""Basic"", ""Live""]}, ""arrival"": {""airport"": {""icao"": ""VOBL"", ""iata"": ""BLR"", ""name"": ""Bangalore"", ""countryCode"": ""in"", ""timeZone"": ""Asia/Kolkata""}, ""scheduledTime"": {""utc"": ""2026-06-29 15:55Z"", ""local"": ""2026-06-29 21:25+05:30""}, ""revisedTime"": {""utc"": ""2026-06-29 15:45Z"", ""local"": ""2026-06-29 21:15+05:30""}, ""terminal"": ""2"", ""gate"": ""D13"", ""baggageBelt"": ""8A"", ""quality"": [""Basic"", ""Live""]}, ""number"": ""AI 2487"", ""status"": ""CheckIn"", ""codeshareStatus"": ""IsOperator"", ""isCargo"": false, ""aircraft"": {""reg"": ""VT-TQC"", ""modeS"": ""80141E"", ""model"": ""Airbus A320 NEO""}, ""airline"": {""name"": ""Air India"", ""iata"": ""AI"", ""icao"": ""AIC""}}",2026-06-29T12:05:13.248Z,e14f65df3ccdcf5a44bce59afd0eb146
bd8bf4e5-a891-4e3e-aea4-adf8386b3bd4,DEL-BLR,DEL,BLR,Air India,AI,Airbus A320 NEO,AI 2512,0,,2026-06-29T13:30:00.000Z,2026-06-29T16:25:00.000Z,175,111.7,ECONOMY,7,15,false,aerodatabox,200,v1,"{""departure"": {""scheduledTime"": {""utc"": ""2026-06-29 13:30Z"", ""local"": ""2026-06-29 19:00+05:30""}, ""revisedTime"": {""utc"": ""2026-06-29 13:30Z"", ""local"": ""2026-06-29 19:00+05:30""}, ""terminal"": ""3"", ""gate"": ""54"", ""quality"": [""Basic"", ""Live""]}, ""arrival"": {""airport"": {""icao"": ""VOBL"", ""iata"": ""BLR"", ""name"": ""Bangalore"", ""countryCode"": ""in"", ""timeZone"": ""Asia/Kolkata""}, ""scheduledTime"": {""utc"": ""2026-06-29 16:25Z"", ""local"": ""2026-06-29 21:55+05:30""}, ""revisedTime"": {""utc"": ""2026-06-29 16:15Z"", ""local"": ""2026-06-29 21:45+05:30""}, ""terminal"": ""2"", ""gate"": ""D12"", ""baggageBelt"": ""8A"", ""quality"": [""Basic"", ""Live""]}, ""number"": ""AI 2512"", ""status"": ""Expected"", ""codeshareStatus"": ""IsOperator"", ""isCargo"": false, ""aircraft"": {""reg"": ""VT-EXN"", ""modeS"": ""800CC7"", ""model"": ""Airbus A320 NEO""}, ""airline"": {""name"": ""Air India"", ""iata"": ""AI"", ""icao"": ""AIC""}}",2026-06-29T12:05:13.248Z,274c313680c86f64ce12e8